# 11 · Raster change, clouds and missing evidence

**Question:** Can a model distinguish vegetation change from missing pixels?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Compare a map interpretation with a paired-pixel calculation, then quantify which claims the evidence supports.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Predict before you run · 25 minutes
**Learn:** align two raster dates, preserve nodata, compute area from pixel size, and audit a visual claim.
Imagine a cloudy second image. Would treating cloud pixels as zero exaggerate vegetation loss?
These are invented vegetation-index grids, not satellite imagery. Both grids share a local metric frame,
extent, resolution and pixel alignment. Values are dimensionless; no sensor calibration is implied.
Change `loss_threshold` and predict whether the detected area will grow or shrink.


In [ ]:
from experiments import paired_change, grade_answer
size=24; pixel_m=25.; loss_threshold=.20
row,col=np.mgrid[:size,:size]
before=.55+.12*np.sin(col/5)*np.cos(row/6)
after=before.copy(); after[5:13,13:21]-=.30
before[0:3,0:8]=np.nan
after[8:11,10:18]=np.nan  # cloud overlaps some true change
result=paired_change(before,after,pixel_m,loss_threshold)
assert result['valid_pixels']==size*size-48
assert result['loss_pixels']<=result['valid_pixels']
assert result['loss_area_m2']==result['loss_pixels']*pixel_m**2
if loss_threshold==.20:
    assert result['loss_pixels']==49  # 64 changed pixels, 15 hidden by cloud
print('Paired valid pixels:',result['valid_pixels'],'of',size*size)
print('Observed loss area (m²):',result['loss_area_m2'])
print('Cloud-hidden change remains unknown to an observer.')


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,4))
cmap=plt.get_cmap('YlGn').copy(); cmap.set_bad('#bdbdbd')
extent=[0,size*pixel_m,0,size*pixel_m]
for ax,field,title in zip(axes[:2],[before,after],['Earlier index','Later index']):
    image=ax.imshow(field,origin='lower',extent=extent,cmap=cmap,vmin=0,vmax=1)
    ax.set(title=title,xlabel='East (m)',ylabel='North (m)')
fig.colorbar(image,ax=list(axes[:2]),label='Synthetic vegetation index',shrink=.75)
delta_cmap=plt.get_cmap('RdBu').copy(); delta_cmap.set_bad('#bdbdbd')
image=axes[2].imshow(result['delta'],origin='lower',extent=extent,cmap=delta_cmap,vmin=-.35,vmax=.35)
axes[2].set(title='Later minus earlier; gray = unknown',xlabel='East (m)',ylabel='North (m)')
fig.colorbar(image,ax=axes[2],label='Index change',shrink=.75)
Path('exports').mkdir(exist_ok=True)
fig.savefig('exports/11_change.png',bbox_inches='tight'); plt.show()
naive_loss=int(((np.nan_to_num(after)-np.nan_to_num(before)) < -loss_threshold).sum())
print('Incorrect zero-fill loss pixels:',naive_loss,'| paired-valid loss pixels:',result['loss_pixels'])
assert naive_loss >= result['loss_pixels']


## Export only the evidence
Send `11_prompt.json` and optionally `11_change.png` in a fresh Astra conversation.
The JSON provides an accessible numerical alternative to the image: rows run south to north,
columns west to east, and `null` means unobserved. Do not send the notebook or the answer key.
For an image-only discussion, ask where the change occurs and why gray pixels cannot establish loss;
do not claim exact pixel-count accuracy from that image. Score the numerical trial below separately.


In [ ]:
def serial_grid(grid):
    return [[float(v) if np.isfinite(v) else None for v in line] for line in grid]
prompt=dict(task='Compare only pixels valid on both dates. Loss is later minus earlier strictly below negative loss_threshold. Return ONLY JSON with valid_pixels, loss_pixels, loss_area_m2. Do not infer values under clouds.',
            synthetic=True,before=serial_grid(before),after=serial_grid(after),pixel_size_m=pixel_m,
            loss_threshold=loss_threshold,nodata=None,rows='south to north',columns='west to east')
export_json('11_prompt.json',prompt)
expected={key:result[key] for key in ['valid_pixels','loss_pixels','loss_area_m2']}


## Capture, score and discuss
The default below is a **deliberately wrong teaching fixture**, not a model response.
Replace the text with the exact response and fill in provenance after an actual trial.
Keep a copy of the original prompt. Explain why observed area is not an estimate of total hidden change.
**Try:** shift the later grid one pixel. Equal shapes no longer establish geographic alignment.


In [ ]:
response_source='FIXTURE - intentionally wrong zero-fill answer'
model_name='not run'; trial_date='not run'; interface='fixture'
response_text=json.dumps(dict(valid_pixels=size*size,loss_pixels=naive_loss,loss_area_m2=naive_loss*pixel_m**2))
score=grade_answer(response_text,expected,{'loss_area_m2':.01})
display(score)
export_json('11_trial.json',dict(response_source=response_source,model=model_name,date=trial_date,
            interface=interface,prompt=prompt,response_text=response_text,score=score,
            tolerance_m2=.01,reference=expected))


## Extend with Codex or Astra
Implement a one-pixel registration diagnostic and compare it with a real change patch. Keep nodata masked. Add a hand-computed 2×2 test and an all-missing test; do not silently interpolate clouds.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
